# Fundamentals 3 · Baselines, Metrics and Softmax

Before training a model it helps to know what "good" means. This notebook computes the simplest possible
models — a constant prediction — for regression and for classification, the metrics used to compare real
models against them (MSE, $R^2$, cross-entropy), and shows how softmax generalizes the sigmoid.

## Regression: the simple bias regressor

A toy dataset of four points:

| x | y |
|---|---|
|1.5|3.2|
|2.3|2.8|
|3.1|1.9|
|4.5|0.7|

In [1]:
import math
import numpy as np

data = np.array([1.5, 2.3, 3.1, 4.5])
targets = np.array([3.2, 2.8, 1.9, 0.7])

print(f"X: {data}")
print(f"Y: {targets}")

X: [1.5 2.3 3.1 4.5]
Y: [3.2 2.8 1.9 0.7]


The best constant prediction under MSE — the **simple bias regressor** (SBR) — is the mean of the targets:

*Optimal Simple Bias Regressor ($\overline{y}$) = $\frac{1}{m}\sum_{i=1}^{n} y_i$*

In [2]:
targets_sum = np.sum(targets)
targets_size = np.size(targets)
sbr = targets_sum/targets_size

print(f"Sum of the elements of Y: {targets_sum}")
print(f"Size of the array Y: {targets_size}")
print(f"Optimal simple bias regressor: {sbr}")


Sum of the elements of Y: 8.6
Size of the array Y: 4
Optimal simple bias regressor: 2.15


Its MSE is the baseline any useful regressor has to beat:

*MSE(y, $\overline{y}$) = $\frac{1}{m}\sum_{i=1}^{n} (y_i - \overline{y_i})^2$*

In [3]:
squared_error = np.square(targets - sbr)
mse_sbr = np.sum(squared_error) / targets_size

print(f"Mean Squared Error of Optimal Simple Bias Regressor: {mse_sbr}")

Mean Squared Error of Optimal Simple Bias Regressor: 0.9225000000000001


$R^2$ compares a model's MSE with the baseline's. Model A has an MSE of 0.2:

*$R^2$ = $1 - \frac{MSE(y, \hat{y})}{MSE(y, \overline{y})}$*

In [4]:
mse_target_a = 0.2
r_squared = 1 - (mse_target_a/mse_sbr)

print(f"R squared: {r_squared}")

R squared: 0.7831978319783198


Model B has an MSE of 2.0:

*$R^2$ = $1 - \frac{MSE(y, \hat{y})}{MSE(y, \overline{y})}$*

In [5]:
mse_target_b = 2.0
r_squared = 1 - (mse_target_b/mse_sbr)

print(f"R squared: {r_squared}")

R squared: -1.168021680216802


**Which model is better?**

Model A: its $R^2$ of 0.78 means 78.3% of the variance in $y$ is explained by the model's inputs, and 21.7% is
noise the model cannot account for. If the model were predicting house prices, for example, 78.3% of the
variation in price would be accounted for by features such as location and size. Model B has an $R^2$ of
−1.168: it does worse than simply predicting the average for every sample, which points to a serious problem
with the model.

Any other constant does worse than the mean. The median, for example:

In [6]:
# median value
non_opt_sbr = np.median(targets)
print(f"Non-optimal Simple Bias Regressor: {non_opt_sbr}")

squared_error = np.square(targets - non_opt_sbr)
mse_non_opt_sbr = np.sum(squared_error) / targets_size
print(f"MSE of Optimal Simple Bias Regressor: {mse_sbr}")
print(f"MSE of Non-optimal Simple Bias Regressor: {mse_non_opt_sbr}")

Non-optimal Simple Bias Regressor: 2.3499999999999996
MSE of Optimal Simple Bias Regressor: 0.9225000000000001
MSE of Non-optimal Simple Bias Regressor: 0.9624999999999999


## Classification: the simple bias classifier

A small dataset whose targets come from three classes, {A, B, C}:

|   x  | y |
|------|---|
|  1.5 | A |
|  3.2 | B |
|  5.4 | B |
| -0.8 | C |

The best constant prediction under cross-entropy is the **empirical class distribution** — the observed
frequency of each class:

*frequency = $\frac{\text{number of observations for class}}{\text{total number of observations}}$*

*Thus, constant distribution is q = ($\frac{1}{4}$, $\frac{2}{4}$, $\frac{1}{4}$)*

*Simplifies to q = ($\frac{1}{4}$, $\frac{1}{2}$, $\frac{1}{4}$); q = (0.25, 0.5, 0.25)*

Softmax turns a model's raw outputs (logits) into probabilities. For $z = (1, 2, 0)$:

*Softmax(($z_1$, $z_2$, $z_3$)) = $(\frac{e^{z_1}}{\sum_{i=1}^{n} e^{z_i}}, \frac{e^{z_2}}{\sum_{i=1}^{n} e^{z_i}}, \frac{e^{z_3}}{\sum_{i=1}^{n} e^{z_i}})$*

In [7]:
# z values
logits = np.array([1.0, 2.0, 0.0])
# e^z values
exp_logits = np.exp(logits)
# sum of e^z values
sum_exp_logits = np.sum(exp_logits)

softmax_logits = exp_logits/sum_exp_logits
print(f"Softmax of logits: {softmax_logits}")


Softmax of logits: [0.24472847 0.66524096 0.09003057]


Categorical cross-entropy of the prediction $(0.1, 0.6, 0.3)$ for the first record, whose true class is A:

*Categorical Cross-Entropy (CCE) = $-\sum_{i=1}^{n} p_i ln(q_i)$*

In [8]:
# Record 1 (x, y) = (1.5, A)
# A is target which is represented in the form of (A, B, C)
target_cce = np.array([1.0, 0.0, 0.0])
prediction_cce = np.array([0.1, 0.6, 0.3])

each_ce = np.log(prediction_cce) * target_cce
cce = -1 * np.sum(each_ce)
print(f"Categorical Cross-Entropy: {cce}")

Categorical Cross-Entropy: 2.3025850929940455


## Softmax and sigmoid are related

The sigmoid works on scalars (element-wise on a vector), while softmax maps a whole vector to a probability
vector, each output depending on every input. For two classes with logits $z = (z_1, z_2)$, fixing $z_2 = 0$
makes the first softmax output exactly the sigmoid of $z_1$.

If $z_2 = 0$, the logits are $(z_1, 0)$:

*S(z) = $(\frac{e^{z_1}}{e^{0} + e^{z_1}}, \frac{e^{0}}{e^{0} + e^{z_1}})$*  
*$S_1$(z) = $\frac{e^{z_1}}{1 + e^{z_1}}$*

*$S_1$(z) = $\frac{e^{z_1}}{1 + e^{z_1}}$*  
*σ($z_1$) = $\frac{1}{1 + e^{-z_1}}$*

*$S_1$(z) = σ($z_1$)*

*$\frac{e^{z_1}}{1 + e^{z_1}}$ = $\frac{1}{1 + e^{-z_1}}$; cross multiply*

*$e^{z_1}(1 + e^{-z_1}) = 1 + e^{z_1}$*

*$e^{z_1} + e^{-z_1+z_1} = 1 + e^{z_1}$*

*$e^{z_1} + e^{0} = 1 + e^{z_1}$*

*$1 + e^{z_1} = 1 + e^{z_1}$*

Thus, *$S_1$(z) = σ($z_1$)*